# 7.3. Padding and Stride

We saw in the previous chapter how the resulting feature map often has dimensions smaller than our original image. This is a result of how the cross-correlation is defined and computed.

Oftentimes, it makes sense to _pad_ our image along the borders so the resulting feature map has the same dimensions as our original image. Then, we know that each resulting pixel `Y[i, j]` in the feature map corresponds to the cross-correlation of the input and convolution kernel centered at `X[i, j]`. A common padding technique is zero-padding, where we pad zeroes to the edges of our input image symmetrically until the resulting feature map has the same dimensions as our original image. This padding technique has the benefit of not requiring us to allocate additional memory - when we apply our convolution kernel to the edges of our image and some pixel indices are out of range then we simply assume them to be zero.

Another concept with convolutional layers is _stride_ - instead of sliding our kernel 1 pixel at a time horizontally and vertically, we might slide it 2, 3 or more pixels instead, using the same or different strides horizontally and vertically. This can be useful, for example, if our image \(and kernel\) is prohibitively large and we just want to extract the key features without centering our convolution window at each adjacent pixel.

In [1]:
import mindspore

mindspore.set_device(device_target='Ascend', device_id=0)
mindspore.run_check()

/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:146: SyntaxWarning: invalid escape sequence '\c'
  2. In forward, tiling would not split c1 and c0, find c1\c0 based on t2.
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:172: SyntaxWarning: invalid escape sequence '\c'
  1. Forward: tiling would not split c1\c0\h0, find c1\c0\h1\h0 based on t2
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangepiaipro-20t/lib/python3.12/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangep

MindSpore version:  2.8.0
The result of multiplication calculation is correct, MindSpore has been installed on platform [Ascend] successfully!


## 7.3.1. Padding

We define an utility function `comp_conv2d` to compute the shape of our feature map based on the dimensions of our input image and the convolutional layer applied.

In [2]:
import mindspore.nn as nn
import mindspore.ops as ops

def comp_conv2d(conv2d, X):
    X = X.reshape((1, 1) + X.shape) # (height, width) => (1, 1, height, width)
    Y = conv2d(X)
    return Y.reshape(Y.shape[2:]) # (1, 1, height, width) => (height, width)

conv2d = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, pad_mode='pad', padding=1)
X = ops.randn((8, 8))
Y = comp_conv2d(conv2d, X)
Y.shape, X.shape == Y.shape

/usr/local/Ascend/cann-8.5.0/python/site-packages/asc_op_compile_base/asc_op_compiler/ascendc_compile_gen_code.py:161: SyntaxWarning: invalid escape sequence '\w'
  match = re.search(f'{option}=(\w+)', ' '.join(compile_options))


((8, 8), True)

As seen above, when our kernel size is $3 \times 3$, padding our image by 1 pixel in all directions cancels out the reduction in dimension caused by our convolution kernel and the resulting feature map has the same dimensions $8 \times 8$ as our original image.

With MindSpore, we can set the `pad_mode` to `'same'` to avoid having to calculate the required padding manually - the framework automatically figures out the required padding \(1 pixel in all directions\) for the resulting feature map to have the same dimensions as our input image.

In [3]:
conv2d_same = nn.Conv2d(1, 1, kernel_size=3, pad_mode='same')
Z = comp_conv2d(conv2d_same, X)
Z.shape, X.shape == Z.shape

/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: Sy

((8, 8), True)

This works even for kernels with distinct height and width, e.g. $5 \times 3$.

In [4]:
conv2d_5x3 = nn.Conv2d(1, 1, kernel_size=(5, 3), pad_mode='same')
W = comp_conv2d(conv2d_5x3, X)
W.shape, X.shape == W.shape

((8, 8), True)

What about even-sized kernel dimensions? For a $4 \times 4$ kernel, the required padding in all directions becomes $\frac{4 - 1}{2} = \frac{3}{2}$. Since we can't pad by a fraction of a pixel, MindSpore automatically assigns the excess padding to the right/bottom edge as explained below.

1. The top and left edges each receive $\lfloor \frac{4 - 1}{2} \rfloor = \lfloor \frac{3}{2} \rfloor = 1$ pixel of zero padding
1. The bottom and right edges each receive $\lfloor \frac{4 - 1}{2} \rfloor + 1 = 2$ pixels of zero padding

In [5]:
conv2d_4x4 = nn.Conv2d(1, 1, kernel_size=(4, 4), pad_mode='same')
Y_0 = comp_conv2d(conv2d_4x4, X)
Y_0.shape, X.shape == Y_0.shape

((8, 8), True)

## 7.3.2. Stride

Stride is the number of pixels we slide our convolution window by for each adjacent element in the resulting feature map.

In [6]:
conv2d_stride = nn.Conv2d(1, 1, kernel_size=3, pad_mode='same', stride=2)
Y_1 = comp_conv2d(conv2d_stride, X)
Y_1.shape, X.shape

((4, 4), (8, 8))

From the output above, we see that applying a stride of 2 roughly halves the dimensions of our resulting feature map in all directions. Similarly, we can expect a threefold reduction in feature map dimensions with a stride of 3 and a ten-fold reduction with a stride of 10, etc.

As with the kernel size and padding \(if manually specified\), we can also use distinct magnitudes for stride in the horizontal vs. vertical directions. For example, a stride of `(3, 4)` means we shift our window 3 pixels horizontally per convolution and shift 4 pixels downward once we hit the right edge of our input image.

In [7]:
conv2d_stride_3x4 = nn.Conv2d(1, 1, kernel_size=(3, 5), pad_mode='pad', padding=(0, 0, 1, 1), stride=(3, 4))
Y_2 = comp_conv2d(conv2d_stride_3x4, X)
Y_2.shape, X.shape

((2, 2), (8, 8))

## 7.3.3. Summary and Discussion

We saw in this chapter:

1. How the dimensions of our feature map relates to that of the input image and convolution kernel
1. How zero-padding can be useful when we want our feature map to have the same dimensions as our input image
1. How stride can be useful for convolutions involving large images to extract just the key features and save computation